In [19]:
!ls /kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1

config.json		model-00001-of-00002.safetensors  tokenizer_config.json
generation_config.json	model-00002-of-00002.safetensors  tokenizer.json
LICENSE			model.safetensors.index.json	  vocab.json
merges.txt		README.md


In [20]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch

# model_name = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/3b-instruct/1"

# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype=torch.float16,
#     trust_remote_code=True
# )

In [21]:
import json
import re
import torch


file_path = "/kaggle/input/datasets/xmohamedsaber/extractor-result/cv_extracted_results.jsonl"

num_test_cases = 1
start_index = 2


def norm_text(value):
    if value is None:
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def norm_list(values):
    return {norm_text(v) for v in values if norm_text(v)}


def strip_leading_bullets(text):
    text = str(text)
    text = re.sub(r"^[•\-\–\—\*]+\s*", "", text)
    return text.strip()


def strip_pipe_suffix(text):
    text = norm_text(text)
    if not text:
        return ""
    return re.split(r"\s*\|\s*", text)[0].strip()


def strip_known_location_suffix(text):
    text = norm_text(text)
    if not text:
        return ""

    # remove trailing location chunks after pipe first
    text = strip_pipe_suffix(text)

    # optional extra cleanup for degree lines that end with location phrases
    location_patterns = [
        r",\s*egypt$",
        r",\s*cairo$",
        r",\s*alexandria$",
        r",\s*giza$",
    ]
    for pat in location_patterns:
        text = re.sub(pat, "", text).strip()

    return text


def clean_education_institution(text):
    text = strip_leading_bullets(text)
    text = strip_pipe_suffix(text)

    # remove appended date ranges if they survived
    text = re.sub(
        r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?\s+\d{4}\s*[-–]\s*"
        r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?\s+\d{4}\b",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    return norm_text(text)


def clean_education_degree(text):
    text = strip_leading_bullets(text)
    text = strip_known_location_suffix(text)

    # remove labels if they leak in
    text = re.sub(r"^(degree|major|specialization)\s*:\s*", "", text, flags=re.IGNORECASE)
    return norm_text(text)


def experience_key(item):
    return (
        norm_text(item.get("company_name", "")),
        norm_text(item.get("title", "")),
    )


def education_key(item):
    return (
        clean_education_institution(item.get("university_name", "")),
        clean_education_degree(item.get("degree", "")),
    )


def education_key_from_candidate(subfields, fallback_value=""):
    institutions = subfields.get("institution_candidates", []) or []
    degrees = subfields.get("degree_candidates", []) or []

    institution = clean_education_institution(institutions[0]) if institutions else ""
    degree = clean_education_degree(degrees[0]) if degrees else ""

    # fallback: try splitting candidate value if subfields are weak
    if not institution and fallback_value:
        parts = re.split(r"\s*\|\s*", str(fallback_value))
        if len(parts) >= 3:
            # many noisy education values look like:
            # degree | location | institution | date
            degree = degree or clean_education_degree(parts[0])
            institution = institution or clean_education_institution(parts[2])

    return (institution, degree)


def project_key(item):
    return norm_text(item.get("project_name", ""))


def training_key(item):
    return (
        norm_text(item.get("title", "")),
        norm_text(item.get("institution", "")),
    )


def build_parser_indexes(parser_payload):
    idx = {}

    idx["technical_skills"] = norm_list(parser_payload.get("technical_skills", []))
    idx["soft_skills"] = norm_list(parser_payload.get("soft_skills", []))

    lang_values = set()
    for lang in parser_payload.get("languages", []):
        full = norm_text(lang)
        if full:
            lang_values.add(full)
            base = norm_text(re.split(r"[\(\-–|]", str(lang))[0])
            if base:
                lang_values.add(base)
    idx["languages"] = lang_values

    idx["certifications"] = norm_list(parser_payload.get("certifications", []))

    idx["experience"] = {
        experience_key(item)
        for item in parser_payload.get("experience", [])
        if experience_key(item) != ("", "")
    }

    idx["education"] = {
        education_key(item)
        for item in parser_payload.get("education", [])
        if education_key(item) != ("", "")
    }

    idx["projects"] = {
        project_key(item)
        for item in parser_payload.get("projects", [])
        if project_key(item)
    }

    idx["trainings_courses"] = {
        training_key(item)
        for item in parser_payload.get("trainings_courses", [])
        if training_key(item) != ("", "")
    }

    idx["email"] = norm_text(parser_payload.get("email", ""))
    idx["phone_number"] = norm_text(parser_payload.get("phone_number", ""))
    idx["summary"] = norm_text(parser_payload.get("summary", ""))
    idx["linkedin"] = norm_text(parser_payload.get("linkedin", ""))
    idx["github"] = norm_text(parser_payload.get("github", ""))
    idx["website"] = norm_text(parser_payload.get("website", ""))

    return idx


def candidate_already_in_parser(candidate, parser_idx):
    ctype = candidate.get("candidate_type", "")
    value = candidate.get("value", "")
    subfields = candidate.get("subfields", {}) or {}

    norm_value = norm_text(value)

    if ctype == "email":
        return norm_value == parser_idx["email"]

    if ctype == "phone_number":
        return norm_value == parser_idx["phone_number"]

    if ctype == "summary":
        return norm_value and norm_value == parser_idx["summary"]

    if ctype == "technical_skill":
        cleaned = norm_text(re.sub(r"^[^:]+:\s*", "", value))
        return cleaned in parser_idx["technical_skills"]

    if ctype == "soft_skill":
        cleaned = norm_text(re.sub(r"^[^:]+:\s*", "", value))
        return cleaned in parser_idx["soft_skills"]

    if ctype == "language":
        return norm_value in parser_idx["languages"]

    if ctype == "certification":
        return norm_value in parser_idx["certifications"]

    if ctype == "project_block":
        names = subfields.get("project_name_candidates", [])
        proj_name = norm_text(names[0]) if names else norm_value
        return proj_name in parser_idx["projects"]

    if ctype == "training":
        titles = subfields.get("title_candidates", [])
        providers = subfields.get("provider_candidates", [])
        key = (
            norm_text(titles[0]) if titles else "",
            norm_text(providers[0]) if providers else "",
        )
        return key in parser_idx["trainings_courses"]

    if ctype == "education_block":
        key = education_key_from_candidate(subfields, fallback_value=value)
        return key in parser_idx["education"]

    if ctype == "experience_block":
        companies = subfields.get("company_candidates", [])
        titles = subfields.get("title_candidates", [])
        key = (
            norm_text(companies[0]) if companies else "",
            norm_text(titles[0]) if titles else "",
        )
        return key in parser_idx["experience"]

    return False


def dedupe_extracted_results_against_parser(parser_payload, extracted_results):
    parser_idx = build_parser_indexes(parser_payload)
    candidates = extracted_results.get("candidates", [])

    filtered_candidates = []
    removed_candidates = []

    for cand in candidates:
        if candidate_already_in_parser(cand, parser_idx):
            removed_candidates.append(cand)
        else:
            filtered_candidates.append(cand)

    cleaned_results = dict(extracted_results)
    cleaned_results["candidates"] = filtered_candidates

    return cleaned_results, removed_candidates

In [22]:
import json
import re
import torch


file_path = "/kaggle/input/datasets/xmohamedsaber/extractor-result/cv_extracted_results.jsonl"

num_test_cases = 1
start_index = 2


phase3_inputs = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        parser_payload = row.get("parser_payload", {})
        extracted_results = row.get("extracted_results", {})

        if isinstance(parser_payload, str):
            try:
                parser_payload = json.loads(parser_payload)
            except json.JSONDecodeError:
                parser_payload = {}

        if isinstance(extracted_results, str):
            try:
                extracted_results = json.loads(extracted_results)
            except json.JSONDecodeError:
                extracted_results = {}

        cleaned_extracted_results, removed_duplicates = dedupe_extracted_results_against_parser(
            parser_payload,
            extracted_results
        )

        phase3_inputs.append({
            "cv_id": row.get("cv_id"),
            "parser_payload": parser_payload,
            "extracted_results": cleaned_extracted_results,
            "removed_duplicates": removed_duplicates,
        })

selected_cases = phase3_inputs[start_index:start_index + num_test_cases]

results = []

for case in selected_cases:
    cv_id = case["cv_id"]
    parser_payload = case["parser_payload"]
    extracted_results = case["extracted_results"]
    removed_duplicates = case["removed_duplicates"]

    messages = [
        {
            "role": "system",
            "content": """
You are a strict CV delta validator.

Input:
- parser_payload: parsed structured CV JSON
- extracted_results: candidate evidence that has already been pre-filtered to remove items that clearly already exist in parser_payload

OBJECTIVE:
Return ONLY information that is supported by extracted_results but missing from parser_payload.

STRICT RULES:
1. Never restate parser_payload.
2. Only return clearly supported missing data from extracted_results.
3. Do not infer, guess, normalize aggressively, or hallucinate.
4. Reject headers, labels, separators, duplicated content, and malformed fragments.
5. Empty output is valid.

Return strict JSON only in this format:

{
  "recovered_fields": {
    "technical_skills": [],
    "soft_skills": [],
    "languages": [],
    "certifications": [],
    "experience": [],
    "projects": [],
    "education": [],
    "trainings_courses": [],
    "awards": [],
    "achievements": [],
    "activities": [],
    "publications": [],
    "website": "",
    "portfolio": "",
    "military_status": "",
    "availability": "",
    "notice_period": ""
  },
  "audit": {
    "recovered_items": [],
    "rejected_candidates": [],
    "suggested_additions": {}
  }
}
"""
        },
        {
            "role": "user",
            "content": json.dumps({
                "cv_id": cv_id,
                "parser_payload": parser_payload,
                "extracted_results": extracted_results
            }, ensure_ascii=False)
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print(f"\n=== CV {cv_id} ===")
    print("Removed duplicates:", len(removed_duplicates))
    print(response)

    results.append({
        "cv_id": cv_id,
        "parser_payload": parser_payload,
        "extracted_results": extracted_results,
        "removed_duplicates": removed_duplicates,
        "response": response
    })


=== CV 3 ===
Removed duplicates: 34
system

You are a strict CV delta validator.

Input:
- parser_payload: parsed structured CV JSON
- extracted_results: candidate evidence that has already been pre-filtered to remove items that clearly already exist in parser_payload

OBJECTIVE:
Return ONLY information that is supported by extracted_results but missing from parser_payload.

STRICT RULES:
1. Never restate parser_payload.
2. Only return clearly supported missing data from extracted_results.
3. Do not infer, guess, normalize aggressively, or hallucinate.
4. Reject headers, labels, separators, duplicated content, and malformed fragments.
5. Empty output is valid.

Return strict JSON only in this format:

{
  "recovered_fields": {
    "technical_skills": [],
    "soft_skills": [],
    "languages": [],
    "certifications": [],
    "experience": [],
    "projects": [],
    "education": [],
    "trainings_courses": [],
    "awards": [],
    "achievements": [],
    "activities": [],
    "pub